# Notebook 03 — Object Detection & Tracking (YOLOv8)

Trains YOLOv8-small on COCO 2017, fine-tunes on custom indoor data, exports for Jetson TensorRT.

## Cell 03-01 | Mount & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, torch

BASE    = '/content/drive/MyDrive/ARGUS'
DS      = f'{BASE}/datasets'
MODELS  = f'{BASE}/models'
CKPTS   = f'{BASE}/checkpoints'
EXPORTS = f'{BASE}/exports'
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'roboflow', 'supervision'])
from ultralytics import YOLO
print(f'Ultralytics loaded | Device: {DEVICE}')

## Cell 03-02 | Download COCO 2017 Dataset

In [ ]:
COCO_DIR  = f'{DS}/coco'
DONE_FLAG = f'{COCO_DIR}/coco_done.flag'
os.makedirs(COCO_DIR, exist_ok=True)

if os.path.exists(DONE_FLAG):
    print('COCO 2017 already downloaded')
else:
    urls = {
        'train2017.zip'                : 'http://images.cocodataset.org/zips/train2017.zip',
        'val2017.zip'                  : 'http://images.cocodataset.org/zips/val2017.zip',
        'annotations_trainval2017.zip' : 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
    }
    for fname, url in urls.items():
        out = f'{COCO_DIR}/{fname}'
        if not os.path.exists(out):
            subprocess.run(['wget', '-q', '--show-progress', '-O', out, url])
            subprocess.run(['unzip', '-q', out, '-d', COCO_DIR])
    open(DONE_FLAG, 'w').close()
    print(f'COCO 2017 ready at {COCO_DIR}')

tr = f'{COCO_DIR}/train2017'
va = f'{COCO_DIR}/val2017'
if os.path.exists(tr):
    print(f'Train: {len(os.listdir(tr))} images | Val: {len(os.listdir(va))} images')

## Cell 03-03 | Create COCO YAML Config

In [ ]:
import yaml

COCO_YAML   = f'{DS}/coco/coco.yaml'
coco_config = {
    'path': f'{DS}/coco', 'train': 'train2017', 'val': 'val2017', 'nc': 80,
    'names': [
        'person','bicycle','car','motorcycle','airplane','bus','train','truck','boat',
        'traffic light','fire hydrant','stop sign','parking meter','bench','bird','cat',
        'dog','horse','sheep','cow','elephant','bear','zebra','giraffe','backpack',
        'umbrella','handbag','tie','suitcase','frisbee','skis','snowboard','sports ball',
        'kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket',
        'bottle','wine glass','cup','fork','knife','spoon','bowl','banana','apple',
        'sandwich','orange','broccoli','carrot','hot dog','pizza','donut','cake','chair',
        'couch','potted plant','bed','dining table','toilet','tv','laptop','mouse','remote',
        'keyboard','cell phone','microwave','oven','toaster','sink','refrigerator','book',
        'clock','vase','scissors','teddy bear','hair drier','toothbrush'
    ]
}
with open(COCO_YAML, 'w') as f: yaml.dump(coco_config, f, default_flow_style=False)
print(f'COCO YAML written: {COCO_YAML}')

## Cell 03-04 | Train YOLOv8-Small on COCO

In [ ]:
from ultralytics import YOLO
import shutil

model   = YOLO('yolov8s.pt')
results = model.train(
    data='coco.yaml', epochs=50, imgsz=640, batch=16, workers=8, device=0,
    project=f'{CKPTS}/yolov8', name='coco_finetune', pretrained=True,
    optimizer='AdamW', lr0=0.001, lrf=0.01, momentum=0.937, weight_decay=0.0005,
    warmup_epochs=3, cos_lr=True, amp=True, save=True, save_period=10,
    cache=False, val=True, plots=True,
)

best = f'{CKPTS}/yolov8/coco_finetune/weights/best.pt'
shutil.copy(best, f'{MODELS}/detection/yolov8s_coco.pt')
print(f'Best weights saved: {MODELS}/detection/yolov8s_coco.pt')

## Cell 03-05 | Fine-Tune on Custom Indoor Data (Roboflow)

In [ ]:
# Option A: Download from Roboflow
# from roboflow import Roboflow
# rf   = Roboflow(api_key='YOUR_ROBOFLOW_API_KEY')
# proj = rf.workspace('your-workspace').project('argus-indoor')
# ds   = proj.version(1).download('yolov8', location=f'{DS}/custom_detection')

# Option B: Manual — point to your data on Drive
CUSTOM_DET  = f'{DS}/custom_detection'
CUSTOM_YAML = f'{CUSTOM_DET}/data.yaml'

if not os.path.exists(CUSTOM_YAML):
    print('No custom detection data found yet.')
    print(f'Upload YOLOv8-format dataset to: {CUSTOM_DET}')
    print('  custom_detection/data.yaml + train/images + train/labels + valid/images + valid/labels')
else:
    print('Custom detection dataset found — starting fine-tune...')
    model_ft   = YOLO(f'{MODELS}/detection/yolov8s_coco.pt')
    results_ft = model_ft.train(
        data=CUSTOM_YAML, epochs=100, imgsz=640, batch=16, device=0,
        project=f'{CKPTS}/yolov8', name='custom_indoor',
        lr0=0.0001, pretrained=True, amp=True,
    )
    import shutil
    shutil.copy(f'{CKPTS}/yolov8/custom_indoor/weights/best.pt',
                f'{MODELS}/detection/yolov8s_argus_final.pt')
    print('Custom fine-tuned model saved')

## Cell 03-06 | Export YOLOv8 to TensorRT

In [ ]:
from ultralytics import YOLO
import shutil

model_final = YOLO(f'{MODELS}/detection/yolov8s_argus_final.pt')
model_final.export(format='onnx', imgsz=640, half=True, simplify=True, opset=17)

shutil.copy(
    f'{MODELS}/detection/yolov8s_argus_final.onnx',
    f'{EXPORTS}/tensorrt/yolov8s_argus_640.onnx'
)
print(f'ONNX exported: {EXPORTS}/tensorrt/yolov8s_argus_640.onnx')
print('On Jetson: yolo export model=yolov8s_argus_final.pt format=engine imgsz=640 half=True')